In [1]:
import pandas as pd
import numpy as np

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
df=pd.read_csv('/content/drive/MyDrive/SQL project/online_shopping_behaviour.csv')

In [ ]:
df.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,12/1/2010 8:26,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,12/1/2010 8:26,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,12/1/2010 8:26,3.39,17850.0,United Kingdom


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 541910 entries, 0 to 541909
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   Invoice      541910 non-null  object 
 1   StockCode    541910 non-null  object 
 2   Description  540456 non-null  object 
 3   Quantity     541910 non-null  int64  
 4   InvoiceDate  541910 non-null  object 
 5   Price        541910 non-null  float64
 6   Customer ID  406830 non-null  float64
 7   Country      541910 non-null  object 
dtypes: float64(2), int64(1), object(5)
memory usage: 33.1+ MB


In [ ]:
# Summary statistics using .describe()
df.describe(include='all') #by default show only integer value

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
count,541910,541910,540456,541910.000000,541910,541910.000000,406830.000000,541910
unique,25900,4070,4223,NaN,23260,NaN,NaN,38
top,573585,85123A,WHITE HANGING HEART T-LIGHT HOLDER,NaN,10/31/2011 14:41,NaN,NaN,United Kingdom
freq,1114,2313,2369,NaN,1114,NaN,NaN,495478
mean,NaN,NaN,NaN,9.552234,NaN,4.611138,15287.684160,NaN
std,NaN,NaN,NaN,218.080957,NaN,96.759765,1713.603074,NaN
min,NaN,NaN,NaN,-80995.000000,NaN,-11062.060000,12346.000000,NaN
25%,NaN,NaN,NaN,1.000000,NaN,1.250000,13953.000000,NaN
50%,NaN,NaN,NaN,3.000000,NaN,2.080000,15152.000000,NaN
75%,NaN,NaN,NaN,10.000000,NaN,4.130000,16791.000000,NaN


In [ ]:
# Checking if missing data or null values are present in the dataset

df.isnull().sum()

,0
Invoice,0
StockCode,0
Description,1454
Quantity,0
InvoiceDate,0
Price,0
Customer ID,135080
Country,0


In [ ]:

#  Backfill missing Description
stockcode_to_description = (
    df.dropna(subset=["Description"])
      .drop_duplicates(subset=["StockCode"])
      .set_index("StockCode")["Description"]
)

missing_desc_mask = df["Description"].isna()
df.loc[missing_desc_mask, "Description"] = df.loc[missing_desc_mask, "StockCode"].map(
    stockcode_to_description
)

recovered = missing_desc_mask.sum() - df["Description"].isna().sum()
print(f"\nRecovered {recovered} Description values via StockCode lookup")


Recovered 1342 Description values via StockCode lookup


In [ ]:

# Drop rows where Description is still missing
before = len(df)
df = df.dropna(subset=["Description"])
print(f"Dropped {before - len(df)} rows with no recoverable Description")

#  Remove fully duplicate rows
before = len(df)
df = df.drop_duplicates()
print(f"\nRemoved {before - len(df):,} fully duplicate rows")

# Remove non-product StockCodes.
NON_PRODUCT_CODES = ["POST", "DOT", "M", "m", "D", "S", "AMAZONFEE",
                      "CRUK", "PADS", "B", "DCGSSGIRL", "DCGSSBOY"]
before = len(df)
df = df[~df["StockCode"].isin(NON_PRODUCT_CODES)]
print(f"Removed {before - len(df):,} non-product rows (postage, fees, adjustments)")

# Remove rows with Price <= 0.
before = len(df)
df = df[df["Price"] > 0]
print(f"Removed {before - len(df):,} rows with Price <= 0")

# Standardize non-country Country values into one label
df["Country"] = df["Country"].replace(
    {"Unspecified": "Unknown", "European Community": "Unknown"}
)

Dropped 112 rows with no recoverable Description

Removed 5,268 fully duplicate rows
Removed 2,754 non-product rows (postage, fees, adjustments)
Removed 2,384 rows with Price <= 0


In [ ]:
total = len(df)
missing_customer = df["Customer ID"].isna().sum()
negative_qty = (df["Quantity"] < 0).sum()

print("\n--- Data Quality Summary ---")
print(f"Total rows after cleaning: {total:,}")
print(f"Rows with missing Customer ID (kept, not dropped): {missing_customer:,} ({missing_customer/total:.1%})")
print(f"Rows with negative Quantity (returns/cancellations, kept): {negative_qty:,} ({negative_qty/total:.1%})")

df.to_csv('/content/drive/MyDrive/SQL project/online_shopping_behaviour.csv', index=False)


--- Data Quality Summary ---
Total rows after cleaning: 531,392
Rows with missing Customer ID (kept, not dropped): 131,590 (24.8%)
Rows with negative Quantity (returns/cancellations, kept): 8,695 (1.6%)


In [ ]:
df.isnull().sum()

,0
Invoice,0
StockCode,0
Description,0
Quantity,0
InvoiceDate,0
Price,0
Customer ID,131590
Country,0


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 531392 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   Invoice      531392 non-null  object 
 1   StockCode    531392 non-null  object 
 2   Description  531392 non-null  object 
 3   Quantity     531392 non-null  int64  
 4   InvoiceDate  531392 non-null  object 
 5   Price        531392 non-null  float64
 6   Customer ID  399802 non-null  float64
 7   Country      531392 non-null  object 
dtypes: float64(2), int64(1), object(5)
memory usage: 36.5+ MB


# **Connecting Python script to PostgreSQL**

In [ ]:
# install psycopg and sqlalchemy
!pip install psycopg2-binary sqlalchemy

In [ ]:
from sqlalchemy import create_engine


username = "postgres"
password = "1404"
host = "localhost"
port = "5432"
database = "customer_behavior"

engine = create_engine(f"postgresql+psycopg2://{username}:{password}@{host}:{port}/{database}")

table_name = "customer"
df.to_sql(table_name, engine, if_exists="replace", index=False)

print(f"Data successfully loaded into table '{table_name}' in database '{database}'.")